# Smoke test : CDGM-7Q — validation des 3 résistances sur 3 architectures

**Objectif** : avant de committer 50-100h de A100, valider numériquement sur **données synthétiques rapides** (CPU OK) que les mécanismes mathématiques de CDGM-7Q tiennent en pratique.

**Ce qu'on teste** :
- **R1 (Bypass-resistance)** : `∂HR_pred/∂A_dag ≠ 0` sur ensemble de mesure positive
- **R2 (Copy-mode immunity)** : `F_θ ≢ 0` à convergence ; `mu_HR_ablation_ratio < 1`
- **R3 (Rank escape)** : `rank(Cov(predictions ensemble)) > rank(A_dag) = 5`

**3 architectures comparées** :
1. **v1** (seed42 baseline) — channel concat de μ_HR à 3 canaux
2. **CDGM-7Q v1 ORIGINAL** — FiLM bottleneck only + σ_data couplé à A_dag + λ_aux=1e-4
3. **CDGM-7Q v2 RÉPARÉE** — FiLM multi-échelle + σ_data fixe + λ_aux=1e-1 + guidance orthogonal

**Setup synthétique** :
- Field 32×32 single-channel
- Vrai DAG rang-5 sur 6 nodes latents
- μ_HR = decoder(A_dag · h) avec h ∈ R^6
- HR = baseline + μ_HR + résidu_rang_plein

**Compute attendu** : 15-30 min sur CPU, 3-5 min sur GPU.

**Verdict** : table comparative + visualisations + PASS/FAIL par architecture par résistance.

In [ ]:
# === Cell 1 : Imports + setup ===
import os
import math
import json
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Seed: {SEED}')

## Section 1 — Générateur SCM synthétique

On simule le setup seed42 en petit :
- 6 variables LR latentes
- A_dag vrai : matrice 6×6 acyclique, rang effectif 5
- μ_HR = `decoder(A_dag · encoder(LR))` ∈ R^{32×32}
- HR = baseline (bicubic-like) + μ_HR + résidu rang-plein
- Target diffusion : δ = HR − baseline (v4 style) OU δ = HR − baseline − μ_HR (v1 style)

Cela reproduit la pathologie copy-mode : si UNet voit μ_HR en input AND target est petit, F_θ ≈ 0 minimise la loss.

In [ ]:
# === Cell 2 : Générateur SCM synthétique ===

@dataclass
class SCMConfig:
    n_nodes: int = 6
    n_lr_chan: int = 4
    spatial: int = 32
    n_samples: int = 4000
    noise_residual: float = 0.3
    noise_baseline: float = 0.1

def make_true_A_dag(n: int = 6, seed: int = 0) -> torch.Tensor:
    """DAG acyclique 6 nodes avec 5 arêtes 'physiques' (chaîne + branches)."""
    g = torch.Generator().manual_seed(seed)
    A = torch.zeros(n, n)
    # Chaîne : 0 → 1 → 2 → 3 → 4 (analog QG cascade z250→z500→z850→SP_HR)
    for i in range(n - 1):
        A[i, i + 1] = 0.8 + 0.2 * torch.rand(1, generator=g).item()
    # 1 méta-path : 0 → 4
    A[0, 4] = 0.5
    return A

class SCMDataset(Dataset):
    """Génère des paires (LR_features, μ_HR, baseline, HR_target).
    
    Structure imitant seed42 : μ_HR est un champ spatial de rang-5 (dans le sens fonctionnel),
    HR contient en plus un résidu rang-plein.
    """
    def __init__(self, cfg: SCMConfig, A_dag: torch.Tensor, seed: int = 0):
        self.cfg = cfg
        self.A = A_dag.clone()
        n = cfg.n_nodes
        S = cfg.spatial
        g = torch.Generator().manual_seed(seed)
        
        # Encoder LR → graph state (frozen)
        self.W_enc = torch.randn(cfg.n_lr_chan, n, generator=g) * 0.5
        # Decoder graph_state → HR field (frozen)
        # Each node decoded as a low-frequency spatial mode
        x = torch.linspace(-1, 1, S)
        y = torch.linspace(-1, 1, S)
        xx, yy = torch.meshgrid(x, y, indexing='ij')
        self.modes = torch.stack([
            torch.sin((i+1) * math.pi * xx) * torch.cos((i+1) * math.pi * yy)
            for i in range(n)
        ], dim=0)  # [n, S, S]
        
        # Génération des samples
        N = cfg.n_samples
        self.LR = torch.randn(N, cfg.n_lr_chan, generator=g) * 1.5
        # Forward through SCM : h_T = (I + A^T) · enc(LR) propagated through chain
        h0 = self.LR @ self.W_enc  # [N, n]
        # Propagation through DAG : h_T = h0 + A^T h0 + A^T A^T h0 + ...
        h = h0.clone()
        propagated = h0.clone()
        for _ in range(n):  # transitive closure
            h = h @ self.A.T
            propagated = propagated + h
        self.h_T = propagated  # [N, n] DAG latent state
        
        # μ_HR = sum of mode_i × h_T[i]
        # [N, n] @ [n, S*S] → [N, S, S]
        modes_flat = self.modes.reshape(n, -1)  # [n, S*S]
        mu_flat = self.h_T @ modes_flat  # [N, S*S]
        self.mu_HR = mu_flat.reshape(N, S, S).unsqueeze(1)  # [N, 1, S, S]
        # Normalize μ_HR to ~unit std
        self.mu_HR = self.mu_HR / (self.mu_HR.std() + 1e-6) * 0.5
        
        # Baseline = low-pass version + noise
        self.baseline = F.avg_pool2d(self.mu_HR.abs(), 4) * 0.7
        self.baseline = F.interpolate(self.baseline, size=(S, S), mode='bilinear', align_corners=False)
        self.baseline = self.baseline + torch.randn_like(self.baseline) * cfg.noise_baseline
        
        # Résidu rang-plein (haute-fréquence)
        self.residual_hf = torch.randn(N, 1, S, S, generator=g) * cfg.noise_residual
        # Pondération spatiale pour donner structure
        gauss_filter = torch.exp(-((xx**2 + yy**2) / 0.5))
        self.residual_hf = self.residual_hf * gauss_filter.unsqueeze(0).unsqueeze(0)
        
        # HR target
        self.HR = self.baseline + self.mu_HR + self.residual_hf
    
    def __len__(self):
        return self.cfg.n_samples
    
    def __getitem__(self, idx):
        return {
            'LR': self.LR[idx],
            'mu_HR': self.mu_HR[idx],
            'baseline': self.baseline[idx],
            'HR': self.HR[idx],
        }

# Build dataset
cfg = SCMConfig()
A_dag_true = make_true_A_dag(cfg.n_nodes, seed=0)
dataset = SCMDataset(cfg, A_dag_true, seed=SEED)
print(f'Dataset : {len(dataset)} samples, spatial {cfg.spatial}×{cfg.spatial}')
print(f'A_dag rank : {torch.linalg.matrix_rank(A_dag_true).item()}')
print(f'μ_HR effective rank (over batch) : {torch.linalg.matrix_rank(dataset.mu_HR[:100].reshape(100, -1)).item()}')
print(f'HR std={dataset.HR.std():.3f}, mean={dataset.HR.mean():.3f}')
print(f'baseline std={dataset.baseline.std():.3f}')
print(f'μ_HR std={dataset.mu_HR.std():.3f}')
print(f'residual_hf std={dataset.residual_hf.std():.3f}')

# Visualisation
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(dataset.baseline[0, 0].numpy()); axes[0].set_title('baseline'); axes[0].axis('off')
axes[1].imshow(dataset.mu_HR[0, 0].numpy()); axes[1].set_title('μ_HR (rank-5)'); axes[1].axis('off')
axes[2].imshow(dataset.residual_hf[0, 0].numpy()); axes[2].set_title('résidu rang-plein'); axes[2].axis('off')
axes[3].imshow(dataset.HR[0, 0].numpy()); axes[3].set_title('HR target'); axes[3].axis('off')
plt.tight_layout(); plt.show()

## Section 2 — Mini-UNet + 3 mécanismes d'injection FiLM

Petit UNet pour tester rapidement. Variants :
- **Concat** : μ_HR concaténé aux canaux d'input (v1 style)
- **FiLM bottleneck only** : FiLM uniquement au bottleneck (CDGM-7Q original)
- **FiLM multi-scale + spatial** : FiLM à chaque niveau decoder + conditionnement spatialement résolu (CDGM-7Q v2 réparée)

In [ ]:
# === Cell 3 : Mini-UNet ===

class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1),
            nn.GroupNorm(min(8, c_out), c_out),
            nn.SiLU(),
            nn.Conv2d(c_out, c_out, 3, padding=1),
            nn.GroupNorm(min(8, c_out), c_out),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.net(x)

class MiniUNet(nn.Module):
    """UNet 2 down/up niveaux, base 16ch.
    Hooks pour FiLM bottleneck et FiLM multi-scale.
    """
    def __init__(self, c_in=2, c_out=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(c_in, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bot = ConvBlock(base * 2, base * 4)
        self.dec2 = ConvBlock(base * 4 + base * 2, base * 2)
        self.dec1 = ConvBlock(base * 2 + base, base)
        self.out = nn.Conv2d(base, c_out, 1)
        self.down = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.base = base
    
    def encode(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.down(e1))
        b = self.bot(self.down(e2))
        return e1, e2, b
    
    def decode(self, e1, e2, b, film_bot=None, film_dec2=None, film_dec1=None):
        # film_X = (gamma, beta) tuple, optional
        if film_bot is not None:
            g, beta = film_bot
            b = g * b + beta
        d2_in = torch.cat([self.up(b), e2], dim=1)
        d2 = self.dec2(d2_in)
        if film_dec2 is not None:
            g, beta = film_dec2
            d2 = g * d2 + beta
        d1_in = torch.cat([self.up(d2), e1], dim=1)
        d1 = self.dec1(d1_in)
        if film_dec1 is not None:
            g, beta = film_dec1
            d1 = g * d1 + beta
        return self.out(d1)

print('MiniUNet : 2 down/up niveaux, base 16ch')
test = MiniUNet(c_in=2, base=16)
with torch.no_grad():
    x = torch.randn(2, 2, 32, 32)
    e1, e2, b = test.encode(x)
    print(f'enc1={e1.shape}, enc2={e2.shape}, bot={b.shape}')
    out = test.decode(e1, e2, b)
    print(f'out={out.shape}')
    n_params = sum(p.numel() for p in test.parameters())
    print(f'Params: {n_params:,}')

In [ ]:
# === Cell 4 : Conditioning modules pour 3 variants ===

class FiLMBottleneck(nn.Module):
    """CDGM-7Q v1 ORIGINAL : ψ = MLP([A_dag flat ; GAP(μ_HR)]) → γ_bot, β_bot.
    σ-conditionné via sigmoid(2u), u = log(σ/σ_data).
    """
    def __init__(self, n_nodes=6, mu_chan=1, base=16):
        super().__init__()
        c_bot = base * 4
        # ψ input : A_dag.flatten() + GAP(μ_HR) per channel + log(σ/σ_data)
        psi_in = n_nodes * n_nodes + mu_chan + 1
        self.mlp = nn.Sequential(
            nn.Linear(psi_in, 64),
            nn.SiLU(),
            nn.Linear(64, 2 * c_bot),  # γ + β
        )
        self.c_bot = c_bot
        self.n_nodes = n_nodes
    
    def forward(self, A_dag, mu_HR, log_sigma_norm):
        # A_dag: [B, n, n] or [n, n], mu_HR: [B, C, H, W], log_sigma_norm: [B]
        B = mu_HR.shape[0]
        if A_dag.dim() == 2:
            A_flat = A_dag.flatten().unsqueeze(0).expand(B, -1)
        else:
            A_flat = A_dag.flatten(1)
        mu_gap = F.adaptive_avg_pool2d(mu_HR, 1).flatten(1)  # [B, C]
        psi_in = torch.cat([A_flat, mu_gap, log_sigma_norm.unsqueeze(-1)], dim=-1)
        out = self.mlp(psi_in)  # [B, 2 c_bot]
        gamma, beta = out.chunk(2, dim=-1)
        # σ-conditioned gating (sigmoid(2 * log_sigma_norm)) — fort à haut σ
        gate = torch.sigmoid(2 * log_sigma_norm).unsqueeze(-1)
        gamma = 1.0 + gate * gamma
        beta = gate * beta
        # Reshape to [B, c_bot, 1, 1]
        return gamma.unsqueeze(-1).unsqueeze(-1), beta.unsqueeze(-1).unsqueeze(-1)

class FiLMMultiScale(nn.Module):
    """CDGM-7Q v2 RÉPARÉE : FiLM à chaque level decoder avec conditionnement SPATIAL.
    À chaque niveau, un petit CNN produit (γ, β) spatialement résolus depuis [A_dag broadcast, μ_HR resampled].
    """
    def __init__(self, n_nodes=6, mu_chan=1, base=16, spatial=32):
        super().__init__()
        self.n_nodes = n_nodes
        self.base = base
        self.spatial = spatial
        # Pour chaque level : un CNN qui produit γ, β à la résolution de ce level
        # Bottleneck : H/4 × W/4 ; dec2 : H/2 × W/2 ; dec1 : H × W
        self.cnn_bot = self._make_cnn(c_in=n_nodes*n_nodes + mu_chan + 1, c_out=2 * base * 4, spatial=spatial // 4)
        self.cnn_dec2 = self._make_cnn(c_in=n_nodes*n_nodes + mu_chan + 1, c_out=2 * base * 2, spatial=spatial // 2)
        self.cnn_dec1 = self._make_cnn(c_in=n_nodes*n_nodes + mu_chan + 1, c_out=2 * base, spatial=spatial)
    
    def _make_cnn(self, c_in, c_out, spatial):
        return nn.Sequential(
            nn.Conv2d(c_in, 16, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(16, c_out, 1),
        )
    
    def _build_cond_map(self, A_dag, mu_HR, log_sigma_norm, target_spatial):
        B, C, H, W = mu_HR.shape
        # Resample μ_HR to target_spatial
        mu_resampled = F.adaptive_avg_pool2d(mu_HR, target_spatial)
        # Broadcast A_dag to spatial map
        if A_dag.dim() == 2:
            A_flat = A_dag.flatten().unsqueeze(0).expand(B, -1)
        else:
            A_flat = A_dag.flatten(1)
        A_map = A_flat.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, target_spatial, target_spatial)
        # Broadcast log_sigma
        sigma_map = log_sigma_norm.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1).expand(B, 1, target_spatial, target_spatial)
        return torch.cat([A_map, mu_resampled, sigma_map], dim=1)
    
    def forward_level(self, level: str, A_dag, mu_HR, log_sigma_norm):
        if level == 'bot':
            target = self.spatial // 4
            cnn = self.cnn_bot
        elif level == 'dec2':
            target = self.spatial // 2
            cnn = self.cnn_dec2
        elif level == 'dec1':
            target = self.spatial
            cnn = self.cnn_dec1
        else:
            raise ValueError(level)
        cond = self._build_cond_map(A_dag, mu_HR, log_sigma_norm, target)
        gamma_beta = cnn(cond)
        gamma, beta = gamma_beta.chunk(2, dim=1)
        # σ-gate spatial (same as v1)
        gate = torch.sigmoid(2 * log_sigma_norm).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        gamma = 1.0 + gate * gamma
        beta = gate * beta
        return gamma, beta

print('FiLMBottleneck (CDGM-7Q v1) et FiLMMultiScale (v2 réparée) définis.')

## Section 3 — Les 3 architectures comparées

Chaque modèle implémente le forward EDM standard avec preconditioning Karras 2022.
Les 3 variants diffèrent uniquement par :
1. **Comment μ_HR entre** (concat / FiLM bot only / FiLM multi-scale)
2. **Quel target** (δ = HR − baseline OU HR − baseline − μ_HR)
3. **σ_data** (fixe v4 OU fonction de A_dag OU fixe)
4. **λ_aux** (0 / 1e-4 / 1e-1)
5. **Sampler guidance** (none / standard / orthogonal-projected)

In [ ]:
# === Cell 5 : 3 architectures ===

@dataclass
class EDMConfig:
    sigma_data: float = 0.5
    sigma_min: float = 0.02
    sigma_max: float = 80.0
    rho: float = 7.0
    P_mean: float = -1.2
    P_std: float = 1.2

def edm_precond(sigma, sigma_data):
    c_skip = sigma_data ** 2 / (sigma ** 2 + sigma_data ** 2)
    c_out = sigma * sigma_data / (sigma ** 2 + sigma_data ** 2).sqrt()
    c_in = 1.0 / (sigma ** 2 + sigma_data ** 2).sqrt()
    c_noise = sigma.log() * 0.25
    return c_skip, c_out, c_in, c_noise

def edm_loss_weight(sigma, sigma_data):
    return (sigma ** 2 + sigma_data ** 2) / (sigma * sigma_data) ** 2


class ArchV1Concat(nn.Module):
    """v1 baseline seed42 : μ_HR concaténé en input UNet. Target = HR - baseline - μ_HR (petit).
    Pathologie connue : copy-mode.
    """
    def __init__(self, edm_cfg: EDMConfig):
        super().__init__()
        # Input : [x_noisy, baseline, μ_HR] = 3 canaux
        self.unet = MiniUNet(c_in=3, c_out=1, base=16)
        self.edm = edm_cfg
        self.target_mode = 'v1'  # target = HR - baseline - μ_HR
    
    def get_sigma_data(self, A_dag=None):
        return self.edm.sigma_data
    
    def forward(self, x_noisy, sigma, A_dag, mu_HR, baseline):
        sigma_data = self.get_sigma_data(A_dag)
        c_skip, c_out, c_in, c_noise = edm_precond(sigma, sigma_data)
        x_in = c_in.view(-1, 1, 1, 1) * x_noisy
        # Concat : [x_in, baseline, μ_HR]
        unet_in = torch.cat([x_in, baseline, mu_HR], dim=1)
        e1, e2, b = self.unet.encode(unet_in)
        F_out = self.unet.decode(e1, e2, b)
        D_out = c_skip.view(-1, 1, 1, 1) * x_noisy + c_out.view(-1, 1, 1, 1) * F_out
        return D_out, F_out


class ArchCDGM7Qv1(nn.Module):
    """CDGM-7Q v1 ORIGINAL : FiLM bottleneck only, σ_data couplé A_dag, λ_aux=1e-4.
    Cette variante est ce que le red-team a tué.
    """
    def __init__(self, edm_cfg: EDMConfig, n_nodes=6, kappa: float = 0.005):
        super().__init__()
        # Input : [x_noisy, baseline] (PAS μ_HR en concat — c'est ce qui change vs v1)
        self.unet = MiniUNet(c_in=2, c_out=1, base=16)
        self.film = FiLMBottleneck(n_nodes=n_nodes, mu_chan=1, base=16)
        self.edm = edm_cfg
        self.kappa = kappa
        self.G_phys = make_true_A_dag(n_nodes, seed=0).to(DEVICE if torch.cuda.is_available() else 'cpu')
        self.target_mode = 'v1'  # target = HR - baseline - μ_HR (same as v1)
        self.lambda_aux = 1e-4
    
    def get_sigma_data(self, A_dag):
        # σ_data = √(σ²_base + κ ‖A - G_phys‖²)
        diff = (A_dag - self.G_phys.to(A_dag.device)).pow(2).sum()
        return (self.edm.sigma_data ** 2 + self.kappa * diff.item()) ** 0.5
    
    def forward(self, x_noisy, sigma, A_dag, mu_HR, baseline):
        sigma_data = self.get_sigma_data(A_dag)
        c_skip, c_out, c_in, c_noise = edm_precond(sigma, sigma_data)
        x_in = c_in.view(-1, 1, 1, 1) * x_noisy
        unet_in = torch.cat([x_in, baseline], dim=1)  # 2 canaux only
        e1, e2, b = self.unet.encode(unet_in)
        # FiLM at bottleneck only
        log_sigma_norm = torch.log(sigma / sigma_data)
        gamma_b, beta_b = self.film(A_dag, mu_HR, log_sigma_norm)
        F_out = self.unet.decode(e1, e2, b, film_bot=(gamma_b, beta_b))
        D_out = c_skip.view(-1, 1, 1, 1) * x_noisy + c_out.view(-1, 1, 1, 1) * F_out
        # Aux loss term : (1 - ‖γ‖)² + ‖β‖²
        aux = (1.0 - gamma_b.flatten(1).norm(dim=1).mean()).pow(2) + beta_b.pow(2).mean()
        return D_out, F_out, aux


class ArchCDGM7Qv2(nn.Module):
    """CDGM-7Q v2 RÉPARÉE :
    - FiLM multi-scale spatial (Attack 2 repair)
    - σ_data fixe v4 (Attack 1 repair)
    - λ_aux=1e-1 + hard check (Attack 4 repair)
    - Target = HR - baseline (v4 style, not v1 !) -- évite copy-mode trap
    - Guidance orthogonale au sampling (Attack 3 repair)
    """
    def __init__(self, edm_cfg: EDMConfig, n_nodes=6, spatial=32, sigma_data_fixed=0.5):
        super().__init__()
        # Input : [x_noisy, baseline] (2 canaux, comme v4)
        self.unet = MiniUNet(c_in=2, c_out=1, base=16)
        self.film = FiLMMultiScale(n_nodes=n_nodes, mu_chan=1, base=16, spatial=spatial)
        self.edm = edm_cfg
        self.sigma_data_fixed = sigma_data_fixed
        self.target_mode = 'v4'  # target = HR - baseline (NOT minus μ_HR)
        self.lambda_aux = 1e-1
    
    def get_sigma_data(self, A_dag=None):
        return self.sigma_data_fixed  # FIXED, not coupled to A_dag
    
    def forward(self, x_noisy, sigma, A_dag, mu_HR, baseline):
        sigma_data = self.get_sigma_data()
        c_skip, c_out, c_in, c_noise = edm_precond(sigma, sigma_data)
        x_in = c_in.view(-1, 1, 1, 1) * x_noisy
        unet_in = torch.cat([x_in, baseline], dim=1)
        e1, e2, b = self.unet.encode(unet_in)
        # FiLM at every level (bot, dec2, dec1)
        log_sigma_norm = torch.log(sigma / sigma_data)
        film_bot = self.film.forward_level('bot', A_dag, mu_HR, log_sigma_norm)
        film_dec2 = self.film.forward_level('dec2', A_dag, mu_HR, log_sigma_norm)
        film_dec1 = self.film.forward_level('dec1', A_dag, mu_HR, log_sigma_norm)
        F_out = self.unet.decode(e1, e2, b, film_bot=film_bot, film_dec2=film_dec2, film_dec1=film_dec1)
        D_out = c_skip.view(-1, 1, 1, 1) * x_noisy + c_out.view(-1, 1, 1, 1) * F_out
        # Aux loss : penalize γ→1 / β→0 across ALL levels
        aux = 0.0
        for g_b in [film_bot, film_dec2, film_dec1]:
            g, beta = g_b
            aux = aux + (1.0 - g.flatten(1).norm(dim=1).mean()).pow(2) + beta.pow(2).mean()
        return D_out, F_out, aux

print('3 architectures définies : ArchV1Concat, ArchCDGM7Qv1, ArchCDGM7Qv2')

## Section 4 — Training loop unifié

Loss = EDM weighted L2 + λ_aux × auxiliary term (selon architecture).
Stage 1 est synthétique frozen (on connait A_dag_true et μ_HR_true depuis le SCM). On entraîne uniquement Stage 2.

In [ ]:
# === Cell 6 : Training loop ===

def make_target(arch, batch, A_dag):
    """Build δ_target based on arch.target_mode.
    v1 : δ = HR - baseline - μ_HR (petit)
    v4 : δ = HR - baseline (plein)
    """
    if arch.target_mode == 'v1':
        return batch['HR'] - batch['baseline'] - batch['mu_HR']
    elif arch.target_mode == 'v4':
        return batch['HR'] - batch['baseline']
    else:
        raise ValueError(arch.target_mode)

def train_one_epoch(arch, loader, optimizer, A_dag, device, has_aux=False):
    arch.train()
    total_loss = 0.0
    total_aux = 0.0
    n_batches = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        delta_target = make_target(arch, batch, A_dag)
        # Sample σ
        B = delta_target.shape[0]
        sigma = torch.exp(arch.edm.P_mean + arch.edm.P_std * torch.randn(B, device=device))
        eps = torch.randn_like(delta_target)
        x_noisy = delta_target + sigma.view(-1, 1, 1, 1) * eps
        # Forward
        if has_aux:
            D_out, F_out, aux = arch(x_noisy, sigma, A_dag, batch['mu_HR'], batch['baseline'])
        else:
            D_out, F_out = arch(x_noisy, sigma, A_dag, batch['mu_HR'], batch['baseline'])
            aux = torch.tensor(0.0, device=device)
        # EDM loss
        sigma_data = arch.get_sigma_data(A_dag) if has_aux else arch.edm.sigma_data
        w = edm_loss_weight(sigma, sigma_data).view(-1, 1, 1, 1)
        edm_loss = (w * (D_out - delta_target).pow(2)).mean()
        # Total
        if has_aux:
            loss = edm_loss + arch.lambda_aux * aux
        else:
            loss = edm_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(arch.parameters(), 1.0)
        optimizer.step()
        total_loss += edm_loss.item()
        total_aux += aux.item() if has_aux else 0.0
        n_batches += 1
    return total_loss / n_batches, total_aux / n_batches

def train_arch(arch, dataset, A_dag, n_epochs=15, batch_size=32, lr=2e-4):
    """Train an architecture and return loss history."""
    arch = arch.to(DEVICE)
    A_dag = A_dag.to(DEVICE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(arch.parameters(), lr=lr)
    has_aux = hasattr(arch, 'lambda_aux')
    history = {'loss': [], 'aux': []}
    t0 = time.time()
    for epoch in range(n_epochs):
        loss, aux = train_one_epoch(arch, loader, optimizer, A_dag, DEVICE, has_aux=has_aux)
        history['loss'].append(loss)
        history['aux'].append(aux)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  ep {epoch+1}/{n_epochs} loss={loss:.4f} aux={aux:.4f} ({time.time()-t0:.0f}s)')
    return history

print('Train loop défini.')

## Section 5 — Mesures diagnostiques R1, R2, R3

Sur une architecture entraînée, on mesure :

### R1 — Bypass-resistance
- Calculer `‖∂HR_pred/∂A_dag‖` via autograd
- Δ_O3 : ablate A_dag → 0, mesurer changement de prédiction
- **PASS si Δ_O3 > 0.05** (la prédiction dépend non-trivialement de A_dag)

### R2 — Copy-mode immunity
- Mesurer `‖F_θ‖` (l'output U-Net non-précondé)
- Mesurer `μ_HR_ablation_ratio = ‖HR_pred(μ_HR) - HR_pred(0)‖ / ‖HR_pred‖`
- **PASS si μ_HR_ablation_ratio ∈ [0.05, 0.95]** (μ_HR contribue mais ne domine pas)
- **FAIL si > 0.95** (copy-mode)

### R3 — Rank escape
- Générer K=32 samples (ensemble)
- Calculer `rank(Cov(samples))`
- **PASS si rank > 5** (échappe au plateau rank-5 de A_dag)

In [ ]:
# === Cell 7 : Mesures diagnostiques ===

@torch.no_grad()
def denoise_one_step(arch, x_noisy, sigma, A_dag, mu_HR, baseline, has_aux=False):
    if has_aux:
        out = arch(x_noisy, sigma, A_dag, mu_HR, baseline)
        D_out, F_out = out[0], out[1]
    else:
        D_out, F_out = arch(x_noisy, sigma, A_dag, mu_HR, baseline)
    return D_out, F_out


def measure_R1_bypass(arch, sample_batch, A_dag, device, has_aux=False):
    """R1 : ∂HR_pred/∂A_dag norm + Δ_O3 ablation."""
    arch.eval()
    A_dag_var = A_dag.clone().detach().to(device).requires_grad_(True)
    batch = {k: v.to(device) for k, v in sample_batch.items()}
    B = batch["HR"].shape[0] if "HR" in sample_batch else 8
    sigma = torch.full((B,), 0.5, device=device)
    sigma_data = arch.get_sigma_data(A_dag_var) if has_aux else arch.edm.sigma_data
    eps = torch.randn_like(batch['HR'])
    delta_target = make_target(arch, batch, A_dag_var)
    x_noisy = delta_target.detach() + sigma.view(-1, 1, 1, 1) * eps  # detach pour éviter loop
    if has_aux:
        D_out, F_out, _ = arch(x_noisy, sigma, A_dag_var, batch['mu_HR'], batch['baseline'])
    else:
        D_out, F_out = arch(x_noisy, sigma, A_dag_var, batch['mu_HR'], batch['baseline'])
    HR_pred = D_out + batch['baseline'] + (batch['mu_HR'] if arch.target_mode == 'v1' else 0)
    # Gradient norm w.r.t. A_dag
    grad_outputs = torch.autograd.grad(
        HR_pred.sum(), A_dag_var,
        create_graph=False, retain_graph=False,
        allow_unused=True,  # v1_concat does not use A_dag in forward
    )
    grad = grad_outputs[0]
    if grad is None:
        grad_norm = 0.0
    else:
        grad_norm = grad.norm().item()
    
    # Δ_O3 : ablation
    with torch.no_grad():
        A_zero = torch.zeros_like(A_dag)
        if has_aux:
            D_zero, _, _ = arch(x_noisy, sigma, A_zero.to(device), batch['mu_HR'], batch['baseline'])
        else:
            D_zero, _ = arch(x_noisy, sigma, A_zero.to(device), batch['mu_HR'], batch['baseline'])
        HR_pred_zero = D_zero + batch['baseline']
        # Pour archs avec target='v1', il faut aussi tester avec μ_HR_zero
        delta_o3 = (HR_pred - HR_pred_zero).abs().mean() / (HR_pred.abs().mean() + 1e-8)
    return grad_norm, delta_o3.item()


def measure_R2_copy_mode(arch, sample_batch, A_dag, device, has_aux=False):
    """R2 : ‖F_θ‖ et μ_HR_ablation_ratio."""
    arch.eval()
    batch = {k: v.to(device) for k, v in sample_batch.items()}
    A_dag = A_dag.to(device)
    B = batch["HR"].shape[0] if "HR" in sample_batch else 8
    sigma = torch.full((B,), 0.5, device=device)
    sigma_data = arch.get_sigma_data(A_dag) if has_aux else arch.edm.sigma_data
    eps = torch.randn_like(batch['HR'])
    delta_target = make_target(arch, batch, A_dag)
    x_noisy = delta_target + sigma.view(-1, 1, 1, 1) * eps
    # Standard prediction
    with torch.no_grad():
        D_out, F_out = denoise_one_step(arch, x_noisy, sigma, A_dag, batch['mu_HR'], batch['baseline'], has_aux=has_aux)
        if arch.target_mode == 'v1':
            HR_pred = D_out + batch['baseline'] + batch['mu_HR']
        else:
            HR_pred = D_out + batch['baseline']
        # Ablation : μ_HR → 0
        D_out_zero, _ = denoise_one_step(arch, x_noisy, sigma, A_dag, torch.zeros_like(batch['mu_HR']), batch['baseline'], has_aux=has_aux)
        if arch.target_mode == 'v1':
            HR_pred_zero = D_out_zero + batch['baseline']  # no μ_HR added back
        else:
            HR_pred_zero = D_out_zero + batch['baseline']
        # Métrique copy-mode : si μ_HR_ablation ≈ 1, copy-mode confirmé
        mu_ablation_ratio = (HR_pred - HR_pred_zero).abs().mean() / (HR_pred.abs().mean() + 1e-8)
        # F_θ magnitude (UNet output non-précondé, mesure si UNet a appris ou collapsé à 0)
        F_mag = F_out.abs().mean().item()
    return F_mag, mu_ablation_ratio.item()


def measure_R3_rank(arch, sample_batch, A_dag, device, K=32, has_aux=False):
    """R3 : rank(Cov(K samples))."""
    arch.eval()
    batch = {k: v[:8].to(device) for k, v in sample_batch.items()}  # 8 conditioning samples
    A_dag = A_dag.to(device)
    B = batch["HR"].shape[0] if "HR" in sample_batch else 8
    sigma = torch.full((B,), 0.5, device=device)
    sigma_data = arch.get_sigma_data(A_dag) if has_aux else arch.edm.sigma_data
    delta_target = make_target(arch, batch, A_dag)
    predictions = []
    with torch.no_grad():
        for k in range(K):
            eps = torch.randn_like(delta_target)
            x_noisy = delta_target + sigma.view(-1, 1, 1, 1) * eps
            D_out, _ = denoise_one_step(arch, x_noisy, sigma, A_dag, batch['mu_HR'], batch['baseline'], has_aux=has_aux)
            predictions.append(D_out.cpu())
    # Stack : [K, B, 1, H, W] → reshape to [K, B*H*W]
    preds = torch.stack(predictions, dim=0)  # [K, B, 1, H, W]
    K_, B, _, H, W = preds.shape
    flat = preds.reshape(K, -1).float()  # [K, B*H*W]
    # Cov [K, K] via flat @ flat.T  (low rank when K < d)
    Cov = (flat - flat.mean(0, keepdim=True)) @ (flat - flat.mean(0, keepdim=True)).T / (flat.shape[1] - 1)
    # numerical rank with tol
    eigvals = torch.linalg.eigvalsh(Cov)
    eigvals = eigvals.clamp(min=0)
    tol = eigvals.max() * 1e-3
    rank = int((eigvals > tol).sum().item())
    return rank, eigvals.tolist()


def verdict_R1(grad_norm, delta_o3, threshold_delta=0.05):
    if delta_o3 > threshold_delta:
        return 'PASS'
    return 'FAIL'

def verdict_R2(F_mag, mu_ratio, F_min=0.05, mu_low=0.05, mu_high=0.95):
    if F_mag < F_min:
        return 'FAIL (F_θ ≈ 0)'
    if mu_ratio > mu_high:
        return 'FAIL (copy-mode)'
    if mu_ratio < mu_low:
        return 'WARN (μ_HR unused)'
    return 'PASS'

def verdict_R3(rank, threshold=5):
    if rank > threshold:
        return 'PASS'
    return 'FAIL (rank-5 trapped)'

print('Mesures R1, R2, R3 définies.')

## Section 6 — Run experiments : entraînement + diagnostics

On entraîne les 3 architectures dans des conditions identiques (15 epochs, same dataset, same seed) puis on mesure R1, R2, R3 sur chacune.

In [ ]:
# === Cell 8 : Run all 3 architectures ===

edm_cfg = EDMConfig(sigma_data=0.5, sigma_min=0.02, sigma_max=80.0)

# Construct architectures
arch_v1 = ArchV1Concat(edm_cfg)
arch_cdgm_v1 = ArchCDGM7Qv1(edm_cfg, n_nodes=cfg.n_nodes, kappa=0.005)
arch_cdgm_v2 = ArchCDGM7Qv2(edm_cfg, n_nodes=cfg.n_nodes, spatial=cfg.spatial, sigma_data_fixed=0.5)

n_epochs = 15
results = {}

for name, arch, has_aux in [
    ('v1_concat', arch_v1, False),
    ('cdgm7q_v1', arch_cdgm_v1, True),
    ('cdgm7q_v2', arch_cdgm_v2, True),
]:
    print(f'\n=== Training {name} ===')
    n_params = sum(p.numel() for p in arch.parameters())
    print(f'  Params: {n_params:,}')
    print(f'  Target mode: {arch.target_mode}')
    print(f'  σ_data: {arch.get_sigma_data(A_dag_true):.4f}')
    if has_aux:
        print(f'  λ_aux: {arch.lambda_aux}')
    hist = train_arch(arch, dataset, A_dag_true, n_epochs=n_epochs, batch_size=32, lr=2e-4)
    
    print(f'\n  Final loss: {hist["loss"][-1]:.4f}')
    # Measure diagnostics
    # Use a fixed sample batch for fair comparison
    sample_batch_idx = list(range(0, 64))
    sample_batch = {
        k: torch.stack([dataset[i][k] for i in sample_batch_idx])
        for k in ['LR', 'mu_HR', 'baseline', 'HR']
    }
    grad_norm, delta_o3 = measure_R1_bypass(arch, sample_batch, A_dag_true, DEVICE, has_aux=has_aux)
    F_mag, mu_ratio = measure_R2_copy_mode(arch, sample_batch, A_dag_true, DEVICE, has_aux=has_aux)
    rank, eigvals = measure_R3_rank(arch, sample_batch, A_dag_true, DEVICE, K=32, has_aux=has_aux)
    
    v1 = verdict_R1(grad_norm, delta_o3)
    v2 = verdict_R2(F_mag, mu_ratio)
    v3 = verdict_R3(rank)
    
    print(f'\n  Diagnostics :')
    print(f'    R1 (Bypass) : grad_norm={grad_norm:.4f}, Δ_O3={delta_o3:.4f} → {v1}')
    print(f'    R2 (Copy-mode) : ‖F_θ‖={F_mag:.4f}, μ_HR_ablation={mu_ratio:.4f} → {v2}')
    print(f'    R3 (Rank escape) : rank(Cov)={rank} (target>5) → {v3}')
    
    results[name] = {
        'final_loss': hist['loss'][-1],
        'history': hist,
        'R1_grad_norm': grad_norm, 'R1_delta_o3': delta_o3, 'R1_verdict': v1,
        'R2_F_mag': F_mag, 'R2_mu_ratio': mu_ratio, 'R2_verdict': v2,
        'R3_rank': rank, 'R3_eigvals': eigvals, 'R3_verdict': v3,
    }

## Section 7 — Tableau comparatif + verdicts finaux

In [ ]:
# === Cell 9 : Tableau résultats ===

print('=' * 80)
print('VERDICT FINAL — Smoke test CDGM-7Q')
print('=' * 80)
print()
print(f'{"Architecture":<20} {"Loss":<10} {"R1 bypass":<25} {"R2 copy-mode":<25} {"R3 rank":<15}')
print('-' * 100)
for name, r in results.items():
    print(f'{name:<20} '
          f'{r["final_loss"]:<10.4f} '
          f'Δ_O3={r["R1_delta_o3"]:.3f} ({r["R1_verdict"]:<6}) '
          f'μ_abl={r["R2_mu_ratio"]:.3f} ({r["R2_verdict"][:8]:<8}) '
          f'rank={r["R3_rank"]:<3} ({r["R3_verdict"][:6]})')
print()

# Verdict global par architecture
print('Verdict global par architecture :')
for name, r in results.items():
    n_pass = sum(['PASS' in r['R1_verdict'], 'PASS' in r['R2_verdict'], 'PASS' in r['R3_verdict']])
    print(f'  {name:<20} : {n_pass}/3 résistances PASS')

# Décision
print()
v2_passes = sum([
    'PASS' in results['cdgm7q_v2']['R1_verdict'],
    'PASS' in results['cdgm7q_v2']['R2_verdict'],
    'PASS' in results['cdgm7q_v2']['R3_verdict'],
])
v1_concat_passes = sum([
    'PASS' in results['v1_concat']['R1_verdict'],
    'PASS' in results['v1_concat']['R2_verdict'],
    'PASS' in results['v1_concat']['R3_verdict'],
])

print('Décision :')
if v2_passes == 3 and v1_concat_passes < 3:
    print('  ✓ CDGM-7Q v2 réparée PASSE les 3 résistances là où v1 baseline échoue.')
    print('  → Validation numérique des repairs (Attacks 1, 2, 4).')
    print('  → COMMIT vers implementation réelle sur seed42 dataset.')
elif v2_passes >= 2:
    print(f'  ◯ CDGM-7Q v2 passe {v2_passes}/3 résistances.')
    print('  → Investigation requise sur la résistance qui FAIL avant commit.')
else:
    print(f'  ✗ CDGM-7Q v2 passe seulement {v2_passes}/3 résistances.')
    print('  → Les repairs ne suffisent pas. Retour case design.')

In [ ]:
# === Cell 10 : Visualisations ===

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1 : Loss curves
for i, (name, r) in enumerate(results.items()):
    axes[0, i].plot(r['history']['loss'], label='EDM loss')
    if r['history']['aux'][0] != 0 or r['history']['aux'][-1] != 0:
        axes[0, i].plot(r['history']['aux'], label='aux loss', alpha=0.7)
    axes[0, i].set_title(f'{name}\nfinal loss = {r["final_loss"]:.4f}')
    axes[0, i].set_xlabel('epoch')
    axes[0, i].set_ylabel('loss')
    axes[0, i].set_yscale('log')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

# Row 2 : Diagnostic bar charts
metrics = ['Δ_O3 (R1)', 'μ_HR_ablation (R2)', 'rank(Cov) (R3)']
values = {
    name: [r['R1_delta_o3'], r['R2_mu_ratio'], r['R3_rank']]
    for name, r in results.items()
}
colors = {'v1_concat': '#e74c3c', 'cdgm7q_v1': '#f39c12', 'cdgm7q_v2': '#27ae60'}

x = np.arange(3)
width = 0.25
for i, (name, vals) in enumerate(values.items()):
    axes[1, 0].bar(x[0] + (i - 1) * width, vals[0], width, label=name, color=colors[name])
    axes[1, 1].bar(x[0] + (i - 1) * width, vals[1], width, label=name, color=colors[name])
    axes[1, 2].bar(x[0] + (i - 1) * width, vals[2], width, label=name, color=colors[name])

axes[1, 0].axhline(0.05, ls='--', color='gray', alpha=0.5, label='R1 threshold')
axes[1, 0].set_title('R1: Δ_O3 (higher = better bypass-resistance)')
axes[1, 0].set_ylabel('Δ_O3')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].axhline(0.95, ls='--', color='red', alpha=0.5, label='Copy-mode threshold')
axes[1, 1].axhline(0.05, ls='--', color='gray', alpha=0.5, label='Min usage')
axes[1, 1].set_title('R2: μ_HR_ablation (✓ in [0.05, 0.95])')
axes[1, 1].set_ylabel('μ_HR_ablation ratio')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].axhline(5, ls='--', color='red', alpha=0.5, label='rank-5 trap')
axes[1, 2].set_title('R3: rank(Cov(ensemble))')
axes[1, 2].set_ylabel('numerical rank')
axes[1, 2].legend(); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Sauvegarde des résultats
import json
save_path = Path('smoke_cdgm7q_results.json')
serializable = {
    name: {
        'final_loss': r['final_loss'],
        'R1_grad_norm': r['R1_grad_norm'],
        'R1_delta_o3': r['R1_delta_o3'],
        'R1_verdict': r['R1_verdict'],
        'R2_F_mag': r['R2_F_mag'],
        'R2_mu_ratio': r['R2_mu_ratio'],
        'R2_verdict': r['R2_verdict'],
        'R3_rank': r['R3_rank'],
        'R3_verdict': r['R3_verdict'],
    }
    for name, r in results.items()
}
with open(save_path, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'\nRésultats sauvegardés : {save_path.absolute()}')

## Interprétation des résultats attendus

Si le red-team a raison (analytiquement) :

| Architecture | R1 attendu | R2 attendu | R3 attendu | Globale |
|---|---|---|---|---|
| **v1_concat** | FAIL (Δ_O3 faible car copy-mode) | FAIL (μ_HR_ablation ≈ 1) | FAIL (rank ≤ 5) | 0-1/3 |
| **cdgm7q_v1 (original)** | dépend (FiLM bottleneck info dégradée) | possible PASS (target petit + FiLM) mais copy-mode peut subsister | FAIL (rank limité par bottleneck) | 1-2/3 |
| **cdgm7q_v2 (réparée)** | PASS (FiLM multi-scale spatially-resolved) | PASS (target v4 + λ_aux=1e-1 + multi-scale) | PASS (rank > 5) | **3/3** |

**Si v2 passe 3/3 et les autres &lt; 3/3** → les repairs valident le design et on peut commit l'implémentation complète.

**Si v2 ne passe PAS 3/3** → diagnostic exact de quelle résistance casse → repair supplémentaire requis.

**Si v2 passe MAIS v1_concat aussi** → le test synthétique n'est pas assez discriminant → repenser le setup.

## Limites du smoke test

Ce test valide les **mécanismes mathématiques** sur un setup synthétique petit. Il ne valide PAS :
- Les métriques RMSE/Pearson/F1 sur des données réelles ACCESS-CM2
- Le compute total (le ratio test/realité est ~100×)
- La Q_phys préservation sur Stage 1 7-node retrain
- L'interaction avec le dataset complet

Si ce smoke test PASSE pour v2 mais l'implémentation complète FAIL, le problème serait dans :
1. Échelle (le test 32×32 ne capture pas la dynamique 172×179)
2. Q_phys 7-node sparse recovery (jamais testé empiriquement)
3. Norris 2019 transferability (citation à vérifier)

Ces 3 risques restent même après PASS du smoke test.